In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader

from dataset_loaders import MPNNDataset

In [ ]:
device = "cuda"

dataset = MPNNDataset(
    device,
    "star_graph_n3",
    program="graph_coloring",
)

loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [3]:
class ToyMPNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.lin_msg1 = nn.Linear(in_dim, hidden_dim)
        self.lin_msg2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, X, edge_index):
        src, dst = edge_index

        x = X.reshape(X.shape[0] * X.shape[1], X.shape[2])
        print(x)

        # Layer 1 message passing
        m1 = self.lin_msg1(x[src])
        agg1 = torch.zeros(x.size(0), m1.size(1)).to(device)
        agg1.index_add_(0, dst, m1)
        h1 = F.relu(agg1)

        # Layer 2 message passing
        m2 = self.lin_msg2(h1[src])
        agg2 = torch.zeros(x.size(0), m2.size(1)).to(device)
        agg2.index_add_(0, dst, m2)
        h2 = F.relu(agg2)

        return h2

In [4]:
# Hyperparameters
in_dim = 1
hidden_dim = 4
out_dim = in_dim  # number of classes
lr = 0.01
epochs = 50

# Model, loss, optimizer
model = ToyMPNN(in_dim, hidden_dim, out_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)

In [5]:
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    total_loss = 0

    for batch in loader:
        X = batch[0]
        y = batch[1]
        out = model(X, dataset.edge_index)  # forward pass
        # print("out", out, "y", y)
        loss = criterion(out, y)  # compute loss
        loss.backward()  # backward pass
        # for name, param in model.named_parameters():
        #     if param.grad is not None:
        #         print(name, param.grad.abs().mean())
        optimizer.step()  # update weights
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        # pred = out.argmax(dim=1)
        # acc = (pred == y).float().mean()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
Epoch 5, Loss: 0.3747
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
tensor([[0.],
        [0.],
        [1.]], device='cuda:0')
tensor([[0.],
        [0.],
        [0.]], device='cuda:0')
tensor([[0.],
    

/home/agaru/anaconda3/envs/cvf/lib/python3.13/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([1, 3, 1])) that is different to the input size (torch.Size([3, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
